# Análise Preditiva de Tempo de Espera em Atracações Portuárias (Regressão)

## 1. Coleta e Carregamento dos Dados

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

# Download e extração dos dados de 2020 a 2024
!wget https://web3.antaq.gov.br/ea/txt/2024.zip
!unzip -o 2024.zip
!rm 2024.zip

!wget https://web3.antaq.gov.br/ea/txt/2023.zip
!unzip -o 2023.zip
!rm 2023.zip

!wget https://web3.antaq.gov.br/ea/txt/2022.zip
!unzip -o 2022.zip
!rm 2022.zip

!wget https://web3.antaq.gov.br/ea/txt/2021.zip
!unzip -o 2021.zip
!rm 2021.zip

!wget https://web3.antaq.gov.br/ea/txt/2020.zip
!unzip -o 2020.zip
!rm 2020.zip

In [ ]:
atracacao2024 = pd.read_csv("/content/2024Atracacao.txt", sep=";")
carga2024 = pd.read_csv("/content/2024Carga.txt", sep=";")

atracacao2023 = pd.read_csv("/content/2023Atracacao.txt", sep=";")
carga2023 = pd.read_csv("/content/2023Carga.txt", sep=";")

atracacao2022 = pd.read_csv("/content/2022Atracacao.txt", sep=";")
carga2022 = pd.read_csv("/content/2022Carga.txt", sep=";")

atracacao2021 = pd.read_csv("/content/2021Atracacao.txt", sep=";")
carga2021 = pd.read_csv("/content/2021Carga.txt", sep=";")

atracacao2020 = pd.read_csv("/content/2020Atracacao.txt", sep=";")
carga2020 = pd.read_csv("/content/2020Carga.txt", sep=";")

## 2. Pré-processamento e Engenharia de Atributos

In [ ]:
# Concatenação dos dataframes
df_atracacao = pd.concat([atracacao2024, atracacao2023, atracacao2022, atracacao2021, atracacao2020])
df_carga = pd.concat([carga2024, carga2023, carga2022, carga2021, carga2020])

# Seleção de colunas relevantes
df_atracacao = df_atracacao[['Ano', 'Mes', 'IDAtracacao', 'CDTUP', 'IDBerco', 'Complexo Portuário', 'Data Atracação', 'Data Chegada', 'Data Desatracação', 'Data Início Operação', 'Data Término Operação', 'Tipo de Operação', 'Terminal']]
df_carga = df_carga[['IDAtracacao', 'Origem', 'Destino', 'Tipo Operação da Carga', 'Natureza da Carga', 'Sentido', 'VLPesoCargaBruta']]

# Limpeza e conversão de tipos
df_carga['VLPesoCargaBruta'] = pd.to_numeric(df_carga['VLPesoCargaBruta'].str.replace(',', '.'))

# Filtrando apenas por 'Movimentação de Carga'
df_atracacao = df_atracacao[df_atracacao["Tipo de Operação"] == "Movimentação da Carga"]

# Removendo duplicatas
df_carga_semDup = df_carga.drop_duplicates()

# Merge das tabelas
df_merged = pd.merge(df_atracacao, df_carga_semDup, on='IDAtracacao', how='left')

# Engenharia de Atributo: Tempo de Espera
df_merged[["Data Atracação", "Data Chegada"]] = df_merged[["Data Atracação", "Data Chegada"]].apply(pd.to_datetime)
df_merged['Tempo de Espera'] = (df_merged['Data Atracação'] - df_merged['Data Chegada']).dt.total_seconds() / 3600 # em horas

# Tratamento de valores nulos ou negativos no tempo de espera
df_merged.dropna(subset=['Tempo de Espera'], inplace=True)
df_merged = df_merged[df_merged['Tempo de Espera'] >= 0]

## 3. Análise Exploratória de Dados (EDA)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

print(df_merged['Tempo de Espera'].describe())

plt.figure(figsize=(12, 6))
sns.histplot(df_merged['Tempo de Espera'], bins=100, kde=True)
plt.title('Distribuição do Tempo de Espera')
plt.xlabel('Tempo de Espera (horas)')
plt.show()

In [ ]:
# Tratamento de outliers (exemplo: remover esperas > 1 ano)
df_merged = df_merged[df_merged['Tempo de Espera'] < 8760]

plt.figure(figsize=(15, 7))
sns.boxplot(data=df_merged, x='Complexo Portuário', y='Tempo de Espera')
plt.title('Tempo de Espera por Complexo Portuário')
plt.xticks(rotation=45)
plt.ylim(0, df_merged['Tempo de Espera'].quantile(0.95)) # Foco na maioria dos dados
plt.show()

## 4. Preparação para Modelagem

In [ ]:
# Seleção de features para o modelo
features = ['Complexo Portuário', 'Natureza da Carga', 'Tipo Operação da Carga', 'Sentido', 'VLPesoCargaBruta']
target = 'Tempo de Espera'

df_model = df_merged[features + [target, 'Ano']].dropna()

# Separação de dados: treino (até 2023) e teste (2024)
df_train = df_model[df_model['Ano'] < 2024]
df_test = df_model[df_model['Ano'] == 2024]

X_train = df_train[features]
y_train = df_train[target]
X_test = df_test[features]
y_test = df_test[target]

print(f"Tamanho do treino: {len(X_train)} amostras")
print(f"Tamanho do teste: {len(X_test)} amostras")

## 5. Desenvolvimento do Modelo de Regressão

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Definir as colunas categóricas e numéricas
categorical_features = ['Complexo Portuário', 'Natureza da Carga', 'Tipo Operação da Carga', 'Sentido']
numeric_features = ['VLPesoCargaBruta']

# Criar o transformador de pré-processamento
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

# Criar o pipeline com o pré-processador e o modelo
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])

# Treinar o modelo
print("Iniciando o treinamento do modelo...")
model_pipeline.fit(X_train, y_train)
print("Treinamento concluído.")

# Fazer previsões no conjunto de teste
print("Realizando previsões no conjunto de teste...")
y_pred = model_pipeline.predict(X_test)

# Avaliar o modelo
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"\nModelo: RandomForestRegressor")
print(f"Raiz do Erro Quadrático Médio (RMSE): {rmse:.2f} horas")

# Exibir algumas previsões vs. valores reais
df_results = pd.DataFrame({'Real': y_test, 'Previsto': y_pred})
print("\nAmostra de resultados:")
print(df_results.head(10))

## 6. Conclusão e Próximos Passos

O modelo `RandomForestRegressor` foi treinado e avaliado, resultando em um valor de RMSE no conjunto de teste. Este valor representa o erro médio do modelo em horas, indicando a precisão geral das previsões.

Como próximos passos para aprimorar o projeto, sugere-se:

1.  **Otimização de Hiperparâmetros:** Utilizar técnicas como `GridSearchCV` ou `RandomizedSearchCV` para encontrar a melhor combinação de hiperparâmetros para o `RandomForestRegressor`, o que pode levar a uma redução significativa do RMSE.
2.  **Experimentação com Outros Modelos:** Testar algoritmos de Gradient Boosting, como `XGBoost` ou `LightGBM`. Esses modelos frequentemente superam o `RandomForest` em performance para dados tabulares.
3.  **Engenharia de Atributos Adicional:** Criar novas features a partir das existentes. Por exemplo, extrair o dia da semana, a quinzena ou a estação do ano a partir das datas pode capturar sazonalidades que influenciam o tempo de espera.
4.  **Análise de Erros:** Investigar os casos em que o modelo apresenta os maiores erros. Isso pode revelar padrões nos dados que não estão sendo bem capturados e guiar melhorias no pré-processamento ou na seleção de features.